In [1]:
import os
from dotenv import load_dotenv

from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_mistralai import MistralAIEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

/var/folders/f_/1j85lpk92ddfd4c7r8d8rz880000gn/T/ipykernel_9701/2586033791.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

In [8]:
PDF_PATH = Path("../documents/hr_policy.pdf")

loader = PyPDFLoader(str(PDF_PATH))

documents = loader.load()

print(f"Loaded documents: {len(documents)}")

Loaded documents: 16


In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")

Total chunks: 70


In [10]:
embeddings = MistralAIEmbeddings(
    model="mistral-embed"
)

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print(f"Vectors stored: {vector_store.index.ntotal}")

Vectors stored: 70


In [28]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

query = "How many Earned leaves can an employee take?"

retrieved_docs = retriever.invoke(query)

print(f"Retrieved documents: {len(retrieved_docs)}")

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Document {i} ---")
    print(doc.page_content)

Retrieved documents: 1

--- Document 1 ---
ii. Total 12 days of Earned leaves will be granted per year.(Year considered as financial year from April- 
March) 
iii. Employee will get Earned leaves once they get complete probation period, it will be based on 
financial year. 
iv. The employer has rights to terminate Employee(s) without notice and without any payment in lieu  
of notice if Employee is taking more than 13 unplanned leaves (including weekends and official offs) 
without any proper updates (and proofs).


In [24]:
context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

print(context)

ii. Total 12 days of Earned leaves will be granted per year.(Year considered as financial year from April- 
March) 
iii. Employee will get Earned leaves once they get complete probation period, it will be based on 
financial year. 
iv. The employer has rights to terminate Employee(s) without notice and without any payment in lieu  
of notice if Employee is taking more than 13 unplanned leaves (including weekends and official offs) 
without any proper updates (and proofs).


In [25]:
prompt = ChatPromptTemplate.from_template(
    """
You are an HR Policy Question Answering system.

Answer the USER QUESTION using ONLY the HR POLICY CONTEXT.

Rules:

- Answer only the specific question asked.
- Use only facts explicitly stated in the context.
- Do not use outside knowledge.
- Do not make assumptions or infer missing information.
- Do not summarize unrelated policy rules.
- Do not mention related policies unless they are required to answer the question.
- If the context does not contain enough information to answer the question,
  say exactly:
  "I could not find this information in the HR policy."
- Keep the answer short and direct.
- Do not provide reasoning or analysis.

HR POLICY CONTEXT:
{context}

USER QUESTION:
{question}

FINAL ANSWER:
"""
)

In [26]:
llm = ChatGoogleGenerativeAI(
    model=os.getenv("GEMINI_MODEL"),
    temperature=0,
    google_api_key=os.getenv("GEMINI_API_KEY"),
)

In [15]:
rag_chain = (
    prompt | llm | StrOutputParser()
)

In [29]:
# user_query = input("Ask hr policy related question: ")

response = rag_chain.invoke({
    "context": context,
    "question": query
})

print(f"type: {type(response)}")
print(f"\n\nanswer: {response}")

type: <class 'langchain_core.messages.base.TextAccessor'>


answer: Based strictly on the provided HR policy context, an employee is granted a total of 12 days of Earned leaves per year. The policy defines the year for this allocation as the financial year, which runs from April to March.

To understand how and when an employee can take these Earned leaves, the policy outlines the following specific rules and conditions:

1. **Allocation and Total Days:** An employee is granted a total of 12 days of Earned leaves per year. This yearly allocation is strictly aligned with the financial year, which is defined as the period from April to March.

2. **Eligibility and Probation Period:** An employee will not receive these Earned leaves immediately upon joining. Instead, the employee will get their Earned leaves once they have successfully completed their probation period. This allocation is also based on the financial year.

3. **Unplanned Leaves and Termination Policy:** While the policy gr